In [ ]:
from pyspark.sql import SparkSession
# 1. Khởi tạo Spark Session kết nối đến Master chung của cụm
spark = SparkSession.builder \
    .appName("MetroPT3_Machine_Learning") \
    .master("spark://10.125.222.18:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()
# 2. Đường dẫn nạp dữ liệu sạch đã lưu trên HDFS
HDFS_ML = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
df = spark.read.parquet(HDFS_ML)
# 4. Kiểm tra cấu trúc dữ liệu để bắt đầu làm Classification
df.printSchema()
print(f"Tổng số bản ghi: {df.count()}")
df.createOrReplaceTempView('sensor')

In [ ]:
# PHÂN TÍCH KHUNG GIỜ CAO ĐIỂM VÀ THẤP ĐIỂM CỦA HỆ THỐNG
q2_combined = spark.sql('''
    SELECT * FROM ( SELECT
            hour,
            ROUND(AVG(COMP)*100, 1) AS phan_tram_chay,
            'CAO_DIEM' AS nhom_tai
        FROM sensor
        GROUP BY hour
        ORDER BY phan_tram_chay DESC
        LIMIT 3
    ) high_load
    UNION ALL
    SELECT * FROM ( SELECT
            hour,
            ROUND(AVG(COMP)*100, 1) AS phan_tram_chay,
            'THAP_DIEM' AS nhom_tai
        FROM sensor
        GROUP BY hour
        ORDER BY phan_tram_chay ASC
        LIMIT 3
    ) low_load
    ORDER BY phan_tram_chay DESC''')
print("EXECUTION PLAN")
q2_combined.explain(True)
print("PHÂN TÍCH 3 KHUNG GIỜ CAO ĐIỂM & 3 KHUNG GIỜ THẤP ĐIỂM")
df_q2 = q2_combined.toPandas()
try:  display(df_q2)
except NameError: print(df_q2.to_string(index=False))